# conv-channel-sum — faded example 3: Use einops.reduce to sum the IC axis of a broadcast product

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-channel-sum`. Running the beacon reports progress on the `CNN: Channel-axis sum semantics` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Channel-axis sum semantics` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-channel-sum`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-channel-sum"
DD_SUBTOPIC = "CNN: Channel-axis sum semantics"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

For a 1x1 conv, every output channel is `sum_ic weight[oc, ic] * x[b, ic, h, w]`. We can form the full `(B, OC, IC, H, W)` product tensor by broadcasting, then collapse the IC axis with `einops.reduce(..., 'b oc ic h w -> b oc h w', 'sum')`. The reduce over `ic` is exactly the channel sum.

## Faded exercise 3

### Faded — reduce the IC axis

Implement `pointwise_via_reduce(x, weight)` for a 1x1 kernel `weight: (OC, IC, 1, 1)`. The broadcasted product tensor `prod` of shape `(B, OC, IC, H, W)` is built for you. Complete the final step that reduces `prod` over the IC axis with `einops.reduce` to produce `(B, OC, H, W)`.

**Fill in:** The reduction of the broadcast product tensor over the IC axis via einops.reduce with the 'sum' op.

In [ ]:
import torch.nn.functional as F

def pointwise_via_reduce(x, weight):
    W2d = rearrange(weight, 'oc ic 1 1 -> oc ic')
    # broadcast: (1, OC, IC, 1, 1) * (B, 1, IC, H, W) -> (B, OC, IC, H, W)
    prod = W2d[None, :, :, None, None] * x[:, None, :, :, :]
    y = None  # TODO: reduce prod over the IC axis (sum) to (B, OC, H, W)
    return y

t.manual_seed(0)
x = t.randn(2, 5, 4, 4)
weight = t.randn(3, 5, 1, 1)
print(pointwise_via_reduce(x, weight).shape)


def _test():
    import torch.nn.functional as F
    t.manual_seed(0)
    x = t.randn(2, 5, 4, 4)
    weight = t.randn(3, 5, 1, 1)
    got = pointwise_via_reduce(x, weight)
    ref = F.conv2d(x, weight)
    assert got.shape == ref.shape, (got.shape, ref.shape)
    assert t.allclose(got, ref, atol=1e-4), (got - ref).abs().max()


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn.functional as F

def pointwise_via_reduce(x, weight):
    W2d = rearrange(weight, 'oc ic 1 1 -> oc ic')
    # broadcast: (1, OC, IC, 1, 1) * (B, 1, IC, H, W) -> (B, OC, IC, H, W)
    prod = W2d[None, :, :, None, None] * x[:, None, :, :, :]
    y = reduce(prod, 'b oc ic h w -> b oc h w', 'sum')
    return y

t.manual_seed(0)
x = t.randn(2, 5, 4, 4)
weight = t.randn(3, 5, 1, 1)
print(pointwise_via_reduce(x, weight).shape)
```
</details>